<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

In [ ]:
%pip install mistralai python-dotenv

In [17]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

load_dotenv()

if "MISTRAL_API_KEY" not in os.environ:
    raise ValueError("MISTRAL_API_KEY not found in environment variables")

mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  2. Working with Different Modalities
</h2>

**Objectives**: Expand knowledge gained in the previous notebook to work with additional modalities than just text

**Difficulty**: 🟢 ⚪️ ⚪️ ⚪️ - Only some basic Python knowledge is required (imports, lists, dict, loops, etc). 

We are building on the previous notebook so you may want to check it out if you're new to the Mistral Python SDK.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.1 Introduction to Content Chunks
</h3>

So far, all our user messages contained the simplest form of content: a string.

```python
    # Familiar syntax
    msg = UserMessage(content="How to reverse a Python list?")

    # Under the hood, this is equivalent to:
    msg = UserMessage(content=[{"type": "text", "text": "How to reverse a Python list?"}])
```
<br>

`content` can either contain a string or a list of **structured content chunks** of different **modalities** (text, audio, image, etc).


**Content** chunks can either be represented with a dict or a class-based notation. Very similar to what you already know for messages.

```python
    # dict notation
    {"type": "text", "text": "What is on this image?"}

    # class-based notation
    from mistralai.client.models import TextChunk
    TextChunk(text="What is on this image?")
```
<br>
The different possible types have been found in the source code (cf. contentchunk.py):<br><br>


  Class | Description |
 |----------|----------|
 | TextChunk(text) | `TextChunk(text="What is on this image?")`|
 | ImageURLChunk(image_url) | `image_url` can either be <br>- **A Web URL** (e.g. "https://upload.wikimedia.org/wikipedia/commons/d/de/Flag_of_the_United_States.png")<br>- **A base64 Encoded Image** (e.g. f"data:image/png;base64,{base64_image}")|
 | DocumentURLChunk(document_url) | `document_url` can either be:<br> - **A Web URL** (e.g. "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf") <br>- **A base64 Encoded Document** (e.g. f"data:application/pdf;base64,{base64_doc}")<br><br>For the most complicated use-cases, the optimized `ocr` endpoint is more appropriate.|
 | AudioChunk(input_audio) | A base64 encoded audio file (e.g. `AudioChunk(input_audio=base64_audio)`<br><br>For transcription, speeach generation and other advanced use-cases, the optimized `audio` endpoint is more appropriate.|
 | FileChunk(file) | For [Studio files](https://console.mistral.ai/build/files): they come from Document AI/OCR, Batches, Audio processing, etc and can also be uploaded via `mistral.files.upload()`|
 | ReferenceChunk(reference) | Citation of the document used for a response - Ignored for now but more info available in the [documentation](https://docs.mistral.ai/studio-api/conversations/citations)|
 | ThinkChunk(thinking) | Thinking traces when `reasoning` is set to `high` in `mistral.chat.complete()` - Ignored for now but more info available in the [documentation](https://docs.mistral.ai/studio-api/conversations/reasoning)|

---

In [18]:
import base64

def encode_file_to_base64(file_path: str) -> str:
    with open(file_path, "rb") as file:
        file_contents = file.read()
    return base64.b64encode(file_contents).decode("utf-8")

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.1 Working with Images
</h3>

In [ ]:
from mistralai.client.models import UserMessage, TextChunk, ImageURLChunk
from IPython.display import Markdown

# Currently allowed formats: JPEG, PNG, WEBP, GIF, MPO, HEIF, AVIF, BMP, TIFF
img_url = r"https://upload.wikimedia.org/wikipedia/commons/d/de/Flag_of_the_United_States.png"

# Create a message with 2 modalities: text + image URL
msg = UserMessage(
    content=
    [
        TextChunk(text="What is on this image?"),
        ImageURLChunk(image_url=img_url)
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)


The image is of the flag of the United States of America. It consists of 13 horizontal stripes alternating in red and white, representing the original 13 colonies. In the canton (the upper left corner), there is a blue rectangle with 50 white stars, each star representing a state in the Union. The stars are arranged in nine offset rows of six stars (top and bottom) and five stars (middle), where rows of six stars are staggered horizontally in relation to the rows of five stars.

In [20]:
# You can use the same technic to work with a local picture but need to convert it to base64
base64_image = encode_file_to_base64(r"images/free api.png")

msg = UserMessage(
    content=
    [
        TextChunk(text="What is on this image?"),
        ImageURLChunk(image_url=f"data:image/png;base64,{base64_image}")
    ]
)

response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

Markdown(response.choices[0].message.content)

The image appears to be a screenshot of a user interface related to subscriptions for a service called "Vibe." Here's a detailed summary:

1. **Title**: The title at the top of the section is "Subscriptions."

2. **Subscriptions List**:
   - There are two subscriptions listed:
     - **Vibe**: This subscription is represented with an orange icon.
     - **API Plan**: This subscription is represented with a blue icon.

3. **Free Mode Section**:
   - **Header**: The section is titled "Free mode."
   - **Description**: The text under the header explains that users can create API keys and use the free tier within the limits described on the limits page. This free usage is included in the Vibe subscription if the user has one, or it is available by default.
   - **Upgrade Button**: There is a prominent black "Upgrade" button on the right side of the free mode section.

This interface seems to be part of a web application where users can manage their subscriptions and understand their usage limits and options for upgrading their plans.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.1 Working with Documents
</h3>

In [21]:
from mistralai.client.models import DocumentURLChunk

doc_url = r"https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"

# Create a message with 2 modalities: text + URL to a PDF file
msg = UserMessage(
    content=
    [
        TextChunk(text="What is this document about?"),
        DocumentURLChunk(document_url=doc_url)
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

This document introduces the **Transformer**, a novel neural network architecture for sequence transduction tasks, such as machine translation, that relies entirely on **attention mechanisms** instead of recurrent or convolutional layers. The paper, titled *"Attention Is All You Need"* by Ashish Vaswani et al., was presented at **NeurIPS 2017**.

### **Key Contributions:**
1. **Architecture Overview:**
   - The Transformer replaces recurrent and convolutional layers with **self-attention** and **multi-head attention**, enabling parallelization and reducing training time.
   - It consists of an **encoder** and **decoder**, both built from stacked layers of self-attention and feed-forward networks.
   - **Positional encodings** (sinusoidal or learned) are added to input embeddings to retain sequence order information.

2. **Attention Mechanisms:**
   - **Scaled Dot-Product Attention:** Computes attention weights by scaling dot products of queries and keys, followed by a softmax over values.
   - **Multi-Head Attention:** Splits attention into multiple heads, allowing the model to jointly attend to different representation subspaces.

3. **Advantages Over Recurrent/Convolutional Models:**
   - **Parallelization:** Unlike RNNs, the Transformer processes all sequence positions simultaneously.
   - **Long-Range Dependencies:** Self-attention connects all positions in constant time, improving learning of distant relationships.
   - **Efficiency:** Achieves state-of-the-art results with significantly less training time (e.g., 3.5 days on 8 GPUs for English-to-French translation).

4. **Experimental Results:**
   - **English-to-German (WMT 2014):** Achieved **28.4 BLEU**, surpassing prior models (including ensembles) by over 2 BLEU.
   - **English-to-French (WMT 2014):** Achieved **41.0 BLEU**, outperforming single models at a fraction of the training cost.
   - Ablation studies show the importance of multi-head attention, residual connections, and dropout.

5. **Future Directions:**
   - Extending the Transformer to other modalities (e.g., images, audio).
   - Exploring restricted attention for efficiency with large inputs.

### **Impact:**
The Transformer became a foundational architecture in deep learning, influencing models like **BERT**, **GPT**, and **Vision Transformers (ViT)**. Its efficiency and scalability have made it the default choice for many NLP tasks.

The paper’s code is available at [Tensor2Tensor](https://github.com/tensorflow/tensor2tensor).

In [22]:
# You can use the same technic to work with a local document but need to convert it to base64
base64_doc = encode_file_to_base64(r"files/2509.18076v1.pdf")

msg = UserMessage(
    content=
    [
        TextChunk(text="What is this document about?"),
        DocumentURLChunk(document_url=f"data:application/pdf;base64,{base64_doc}")
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

The document presents a study on improving the function-calling capabilities of large language models (LLMs) by introducing a structured, template-based reasoning framework. The authors argue that free-form Chain-of-Thought (CoT) prompting is insufficient for structured function-calling tasks, leading to errors in tool selection, parameterization, and user intent interpretation. To address this, they propose a curriculum-inspired framework that guides LLMs through deliberate step-by-step reasoning using structured templates. This approach aims to enhance the accuracy, interpretability, and transparency of tool-using agents.

The methodology involves two main components: prompting strategies and fine-tuning strategies. For prompting, the authors design a structured template that breaks down the function-calling process into critical stages, such as identifying relevant functions, examining documentation, extracting parameters, and validating the final function call. For fine-tuning, they construct a synthetic dataset called ToolGT, which systematically encodes reasoning patterns using structured templates. This dataset is designed to teach models to adhere to formatting conventions, execute step-by-step reasoning, and align outputs with API specifications.

The experimental results demonstrate that the proposed template-based prompting and fine-tuning methods consistently outperform both No-Thought and CoT approaches across various models and benchmarks, including BFCLv2 and Nexus. The authors highlight that their method reduces tool-use errors and improves robustness, interpretability, and transparency. They also discuss limitations, such as the need for more comprehensive datasets covering complex scenarios and the potential for adaptive template strategies.

Overall, the document emphasizes the importance of structured reasoning in enhancing the reliability and interpretability of LLMs for real-world tool interactions.

In [ ]:
from mistralai.client.models import FileChunk
from IPython.display import display

# For Studio files (https://console.mistral.ai/build/files): they come from Document AI/OCR, Batches, Audio processing, etc and can also be uploaded via `mistral.files.upload()`
files = mistral.files.list().data

if len(files)>0:
    first_file_id=files[0].id

    msg = UserMessage(
        content=
        [
            FileChunk(file_id=first_file_id),
            TextChunk(text="What is this file about?")
        ]
    )

    response = mistral.chat.complete(
        model="mistral-small-latest",
        messages=[msg]
    )

    display(Markdown(response.choices[0].message.content))

else:
    print("No files available")

The document provides a step-by-step guide on how to access and utilize the LangChain Documentation MCP Server to retrieve up-to-date information about LangChain. It outlines the process of selecting connectors, adding the MCP server, enabling specific functions, and then using the connector to ask questions. The document includes visual steps and descriptions for each stage, such as selecting the connector directory, adding the MCP server with its URL and description, and enabling functions like 'Search Docs By LangChain' and 'Query Docs Filesystem Docs By LangChain'. It also demonstrates how to interact with the system by asking a question about creating a Python agent in LangChain, showing the expected output and a minimal example of the code required.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  2.1 Working with Audio
</h3>

In [52]:
from mistralai.client.models import AudioChunk

base64_audio = encode_file_to_base64(r"audio/boulangerie.m4a")

msg = UserMessage(
    content=
    [
        AudioChunk(input_audio=base64_audio),
        TextChunk(text="Transcribe this audio file into English please and tell me what language is spoken in the audio.")
    ]
)

# Request completion
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[msg]
)

# Display response
Markdown(response.choices[0].message.content)

The English transcription of the audio file is:

**"Hello, so one baguette, two croissants, and one lemon tart, please. That will be all, thank you."**

The language spoken in the audio is **French**.

In [ ]:
files = mistral.files.list().data
first_file_id=files[0].id

[FileSchema(id='7b0c2364-d76c-4edd-ae3c-1c9aec2338ff', object='file', size_bytes=533447, created_at=1780849699, filename='LangChain Documentation MCP Server.pdf', purpose='ocr', sample_type='ocr_input', source='upload', num_lines=0, mimetype='application/pdf', signature=None, expires_at=1780853299, visibility='user'),
 FileSchema(id='3bf7fe22-b7b3-41f3-aa5e-9a0be1fca916', object='file', size_bytes=533447, created_at=1780849679, filename='LangChain Documentation MCP Server.pdf', purpose='ocr', sample_type='ocr_input', source='upload', num_lines=0, mimetype='application/pdf', signature=None, expires_at=1780853278, visibility='user')]

In [9]:
files[0].id

'7b0c2364-d76c-4edd-ae3c-1c9aec2338ff'